# FLAG 2027 - Notebook 14: MAV-Celeb v2 (84 Indian celebrities, organizer-approved) + v1 held-out ids

More in-domain identities gave our biggest gain so far (v1+v3: 31.4 -> 28.7). The organizers approved v2 (English-Hindi) on 21 Oct.

1. Unpack `mavceleb_v2_complete.zip` (+ the v1 identities that were NOT in the v1 train list from `mavceleb_v1_complete.zip`), build one row per voice clip with a face frame of the same person, drop any Bengali clip (language-ID, to be safe).
2. Extract CLIP / ECAPA (official) / WavLM / ResNet293 - exactly as for our other data.
3. Train our two best recipes WITH the new people: **N4x** (challenge + all MAV-Celeb) and **B1x** (+ VoxCeleb2, bal75). CV + submission ZIPs + an ensemble ZIP.

First run about 2-3 h. Resumable. Paste the **SUMMARY** to Claude. If Cell 3 stops with 'unexpected zip layout', paste its output.

In [ ]:
# Cell 1 - Drive, GPU, packages
from google.colab import drive
drive.mount('/content/drive')
!pip -q install speechbrain onnx onnx2torch
import torch, shutil
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', dev, '| disk free %.0f GB' % (shutil.disk_usage('/content')[2] / 1e9))

In [ ]:
# Cell 2 - paths + encoders (official ECAPA, language-ID, CLIP, WavLM, ResNet293) - same as Notebooks 06/07/13
import os, re, json, time, zipfile, copy
import numpy as np, pandas as pd
import librosa, soundfile as sf
from PIL import Image
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download, list_repo_files

BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027')
FEATDIR = os.path.join(PROJECT, 'features')
RESDIR = os.path.join(PROJECT, 'results')
CACHE = os.path.join(PROJECT, 'models_cache')
MAV_ZIPS = os.path.join(BASE, 'dataset')
MAV_LISTS = os.path.join(BASE, 'face_voice_association_splits')
SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)

try:
    from speechbrain.inference.interfaces import Pretrained
    from speechbrain.inference.classifiers import EncoderClassifier
except ImportError:
    from speechbrain.pretrained.interfaces import Pretrained
    from speechbrain.pretrained import EncoderClassifier
class Encoder(Pretrained):
    MODULES_NEEDED = ['compute_features', 'mean_var_norm', 'embedding_model']
    def encode_batch(self, wavs, wav_lens=None):
        if wav_lens is None:
            wav_lens = torch.ones(wavs.shape[0], device=self.device)
        wavs, wav_lens = wavs.to(self.device).float(), wav_lens.to(self.device)
        return self.mods.embedding_model(self.mods.mean_var_norm(self.mods.compute_features(wavs), wav_lens), wav_lens)
ECAPA = Encoder.from_hparams(source='yangwang825/ecapa-tdnn-vox2', savedir=os.path.join(CACHE, 'ecapa_vox2'), run_opts={'device': dev}); ECAPA.eval()
LID = EncoderClassifier.from_hparams(source='speechbrain/lang-id-voxlingua107-ecapa', savedir=os.path.join(CACHE, 'lid_voxlingua107'), run_opts={'device': dev})

def batches_by_length(paths, bs):
    order = np.argsort([sf.info(p).duration for p in paths])
    for i in range(0, len(order), bs):
        yield order[i:i + bs]
def pad(paths, idx, max_sec=None):
    wavs = [librosa.load(paths[j], sr=16000, mono=True)[0] for j in idx]
    if max_sec:
        wavs = [w[:16000 * max_sec] for w in wavs]
    lens = torch.tensor([len(w) for w in wavs], dtype=torch.float); L = int(lens.max())
    b = torch.zeros(len(wavs), L)
    for k, w in enumerate(wavs):
        b[k, :len(w)] = torch.from_numpy(w)
    return b, lens / L
def enc_ecapa(paths, bs=16):
    out = np.zeros((len(paths), 192), np.float32)
    for idx in tqdm(list(batches_by_length(paths, bs)), desc='ecapa'):
        b, rl = pad(paths, idx)
        with torch.no_grad():
            out[idx] = ECAPA.encode_batch(b, rl).squeeze(1).cpu().numpy()
    return out
def language_of(paths, bs=32):
    out = [''] * len(paths)
    for idx in tqdm(list(batches_by_length(paths, bs)), desc='language-ID'):
        b, rl = pad(paths, idx, max_sec=10)
        with torch.no_grad():
            labs = LID.classify_batch(b.to(dev), rl.to(dev))[3]
        for j, lab in zip(idx, labs):
            out[j] = lab.split(':')[0].strip()
    return np.array(out)
def enc_clip(paths):
    from transformers import CLIPModel, CLIPProcessor
    name = 'openai/clip-vit-large-patch14'
    model = CLIPModel.from_pretrained(name).half().to(dev).eval(); proc = CLIPProcessor.from_pretrained(name)
    out = []
    for i in tqdm(range(0, len(paths), 64), desc='clip'):
        ims = [Image.open(p).convert('RGB') for p in paths[i:i + 64]]
        x = proc(images=ims, return_tensors='pt')['pixel_values'].to(dev, torch.float16)
        with torch.no_grad():
            out.append(model.visual_projection(model.vision_model(pixel_values=x).pooler_output).float().cpu().numpy())
    return np.concatenate(out)
def enc_wavlm(paths, max_sec=10):
    from transformers import AutoFeatureExtractor, WavLMForXVector
    name = 'microsoft/wavlm-base-plus-sv'
    fe = AutoFeatureExtractor.from_pretrained(name); model = WavLMForXVector.from_pretrained(name).to(dev).eval()
    out = []
    for i in tqdm(range(0, len(paths), 32), desc='wavlm'):
        wavs = [librosa.load(p, sr=16000, mono=True)[0][:16000 * max_sec] for p in paths[i:i + 32]]
        x = fe(wavs, sampling_rate=16000, padding=True, return_attention_mask=True, return_tensors='pt').to(dev)
        with torch.no_grad():
            out.append(model(**x).embeddings.cpu().numpy())
    return np.concatenate(out)
def enc_r293(paths, sec=6, bs=32):
    import torchaudio
    from onnx2torch import convert
    repo = 'Wespeaker/wespeaker-voxceleb-resnet293-LM'
    onnx_name = [f for f in list_repo_files(repo) if f.endswith('.onnx')][0]
    net = convert(hf_hub_download(repo, onnx_name, local_dir=os.path.join(CACHE, 'wespeaker_r293'))).to(dev).eval()
    L = 16000 * sec
    def fbank(p):
        w = librosa.load(p, sr=16000, mono=True)[0]
        if len(w) >= L:
            s0 = (len(w) - L) // 2; w = w[s0:s0 + L]
        else:
            w = np.tile(w, int(np.ceil(L / max(len(w), 1))))[:L]
        fb = torchaudio.compliance.kaldi.fbank(torch.from_numpy(w).unsqueeze(0) * (1 << 15), num_mel_bins=80, frame_length=25,
                                               frame_shift=10, dither=0.0, sample_frequency=16000, window_type='hamming', use_energy=False)
        return fb - fb.mean(0, keepdim=True)
    out = []
    for i in tqdm(range(0, len(paths), bs), desc='resnet293'):
        x = torch.stack([fbank(p) for p in paths[i:i + bs]]).to(dev)
        with torch.no_grad():
            e = net(x)
        e = e[0] if isinstance(e, (list, tuple)) else e
        out.append(e.reshape(len(x), -1).float().cpu().numpy())
    return np.concatenate(out)
print('encoders ready')

In [ ]:
# Cell 3 - unpack v2 (all ids) + v1 held-out ids; one row per voice clip + a face frame of the same person (RESUMABLE)
EXTRA_PATH = os.path.join(FEATDIR, 'mavceleb_extra.npz')
MAX_PER_ID = 120          # clips per person (more PEOPLE matter more than more clips)
VPAT = re.compile(r'voices/(id\d+)/([^/]+)/([^/]+)/([^/]+\.wav)$')
FPAT = re.compile(r'faces/(id\d+)/([^/]+)/([^/]+)/([^/]+\.jpg)$')

# ids already used from v1 (train list) -> only the OTHER v1 ids are new
v1_used = set()
for lf in ['English_train.txt', 'Urdu_train.txt']:
    for l in open(os.path.join(MAV_LISTS, 'v1', lf)):
        t = l.split()
        if t:
            v1_used.add(t[0].split('/')[0])

if not os.path.exists(EXTRA_PATH):
    rows = []   # (version, id, lang, voice path, face path)
    for ver, zname, skip_ids in [('v2', 'mavceleb_v2_complete.zip', set()), ('v1', 'mavceleb_v1_complete.zip', v1_used)]:
        t0 = time.time()
        local = '/content/' + zname
        shutil.copy(os.path.join(MAV_ZIPS, zname), local)
        zf = zipfile.ZipFile(local)
        names = zf.namelist()
        voices, faces = {}, {}
        for n in names:
            mv = VPAT.search(n)
            if mv:
                voices.setdefault(mv.group(1), []).append((mv.group(2), mv.group(3), n)); continue
            mf = FPAT.search(n)
            if mf:
                faces.setdefault((mf.group(1), mf.group(2), mf.group(3)), []).append(n)
        if not voices:
            print('unexpected zip layout - first 25 entries of', zname)
            for n in names[:25]:
                print('   ', n)
            raise SystemExit('unexpected zip layout in %s - paste the lines above to Claude' % zname)
        ids = sorted(i for i in voices if i not in skip_ids)
        rng = np.random.default_rng(0)
        D = '/content/mavx/' + ver
        for pid in ids:
            vs = voices[pid]
            vs = [vs[j] for j in rng.permutation(len(vs))[:MAX_PER_ID]]
            for lang, vid, vn in vs:
                fr = faces.get((pid, lang, vid)) or [f for (p, l, v), fl in faces.items() if p == pid for f in fl]
                if not fr:
                    continue
                fn = fr[rng.integers(len(fr))]     # a face frame of the same person (same video when available)
                rows.append((ver, pid, lang, zf.extract(vn, D), zf.extract(fn, D)))
        zf.close(); os.remove(local)
        note('%s: %d new people, %d rows so far (%.0f min)' % (ver, len(ids), len(rows), (time.time() - t0) / 60))
    ver_, pid_, lang_, vp, fp = map(list, zip(*rows))
    lid = language_of(vp)
    keep = lid != 'bn'
    note('language-ID: Bengali clips removed: %d of %d | top detected: %s' % (int((~keep).sum()), len(keep), dict(pd.Series(lid).value_counts().head(5))))
    vp = [v for v, k in zip(vp, keep) if k]; fp = [f for f, k in zip(fp, keep) if k]
    spk = np.array(['%s_%s' % (a, b) for a, b, k in zip(ver_, pid_, keep) if k])
    lang = np.array([l for l, k in zip(lang_, keep) if k])
    feats = {'spk': spk, 'lang': lang}
    for name, kind, fn_ in [('clip', fp, enc_clip), ('ecapa', vp, enc_ecapa), ('wavlm_sv', vp, enc_wavlm), ('r293', vp, enc_r293)]:
        t0 = time.time(); feats[name] = fn_(kind).astype(np.float32)
        note('extra %-8s extracted in %.1f min' % (name, (time.time() - t0) / 60))
    np.savez(EXTRA_PATH, **feats)
EXTRA = dict(np.load(EXTRA_PATH))
note('MAV-Celeb extra: %d clips, %d people | languages %s' % (len(EXTRA['spk']), len(set(EXTRA['spk'])), dict(pd.Series(EXTRA['lang']).value_counts())))

In [ ]:
# Cell 4 - load challenge + MAV (v1/v3) + MAV extra + VoxCeleb2; CLIP | ECAPA + WavLM + ResNet293
import torch.nn as nn
import torch.nn.functional as Fn
from sklearn.metrics import roc_curve
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
LOCAL = '/content/flag_small'; os.makedirs(LOCAL, exist_ok=True)
for z in ['train_set.zip', 'dev_set.zip']:
    flag = os.path.join(LOCAL, z + '.done')
    if not os.path.exists(flag):
        with zipfile.ZipFile(os.path.join(ZIP_DIR, z)) as zf:
            zf.extractall(LOCAL, members=[m for m in zf.namelist() if m.endswith(('.csv', '.txt'))])
        open(flag, 'w').close()
INDEX = [os.path.join(r, f) for r, _, fs in os.walk(LOCAL) for f in fs if f.endswith(('.csv', '.txt'))]
def find_one(*parts):
    hits = [p for p in INDEX if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]
def read_feat(path, dim):
    df = pd.read_csv(path, header=None)
    return df.iloc[:, :dim].values.astype(np.float32), (df.iloc[:, dim].values if df.shape[1] > dim else None)
SPLIT_NAMES = ['train', 'no_gender_English', 'no_gender_Bangla', 'gender_English', 'gender_Bangla']
FEAT = {}
Xv, y = read_feat(find_one('train_English_voices.csv'), 192)
FEAT['ecapa'] = {'train': Xv}
for s in SPLIT_NAMES[1:]:
    track, lang_ = s.rsplit('_', 1)
    FEAT['ecapa'][s] = read_feat(find_one('/' + track + '/', lang_ + '_test_voices.csv'), 192)[0]
for name in ['clip', 'wavlm_sv', 'r293']:
    d = np.load(os.path.join(FEATDIR, name + '.npz')); FEAT[name] = {s: d[s] for s in d.files}
y = y.astype(int)
train_lines = [l.split() for l in open(find_one('train_English.txt')) if l.strip()]
cls2spk = {int(l[5]): l[4] for l in train_lines}
meta = pd.read_csv(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'), header=None, names=['spk', 'g'])
spk2g = dict(zip(meta.spk.astype(str).str.strip(), meta.g.astype(str).str.strip().str.lower()))
ids = np.unique(y); id_gender = {c: spk2g[cls2spk[c]] for c in ids}
MAV = dict(np.load(os.path.join(FEATDIR, 'mavceleb_v1v3.npz'))); MAV['r293'] = FEAT['r293']['mav']
VOX = {k: [] for k in ['spk', 'clip', 'ecapa', 'wavlm_sv', 'r293']}
for tag in ['vox2_test_mp4', 'vox2_mp4_1']:
    d = dict(np.load(os.path.join(FEATDIR, 'vox_' + tag + '.npz'))); d['r293'] = np.load(os.path.join(FEATDIR, 'vox_' + tag + '_r293.npy'))
    for k in VOX:
        VOX[k].append(d[k])
VOX = {k: np.concatenate(v) for k, v in VOX.items()}

def l2(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
VOICE = ['ecapa', 'wavlm_sv', 'r293']
def face_of(clip):
    return l2(clip).astype(np.float32)
def voice_of(src):
    return np.concatenate([l2(src[n]) for n in VOICE], 1).astype(np.float32)
CH = {n: FEAT[n]['train'] for n in ['clip'] + VOICE}
N_CH, N_MAV, N_EX, N_VOX = len(y), len(MAV['spk']), len(EXTRA['spk']), len(VOX['spk'])
ext_spk = list(MAV['spk']) + list(EXTRA['spk']) + list(VOX['spk'])
ext_lab = {s: 1000 + i for i, s in enumerate(sorted(set(ext_spk)))}
Y_ALL = np.concatenate([y, np.array([ext_lab[s] for s in ext_spk])])
D_ALL = np.r_[np.zeros(N_CH, int), np.ones(N_MAV + N_EX, int), 2 * np.ones(N_VOX, int)]   # MAV extra = same pipeline as MAV -> domain 1
F_RAW = np.concatenate([face_of(CH['clip']), face_of(MAV['clip']), face_of(EXTRA['clip']), face_of(VOX['clip'])])
V_RAW = np.concatenate([voice_of(CH), voice_of(MAV), voice_of(EXTRA), voice_of(VOX)])
ROWS = {'mav_all': N_CH + np.arange(N_MAV + N_EX), 'mav_all+vox': N_CH + np.arange(N_MAV + N_EX + N_VOX)}
note('Table: %d rows, %d people (in-domain people: %d)' % (len(Y_ALL), len(set(Y_ALL)), len(set(Y_ALL[D_ALL < 2]))))

In [ ]:
# Cell 5 - training (N4 recipe without VoxCeleb; B1 = bal75 recipe with VoxCeleb) + CV
def domain_normalize(tr_idx):
    F, V = F_RAW.copy(), V_RAW.copy(); ch = None
    for d in np.unique(D_ALL[tr_idx]):
        trd = tr_idx[D_ALL[tr_idx] == d]; alld = np.where(D_ALL == d)[0]
        st = (F_RAW[trd].mean(0), F_RAW[trd].std(0) + 1e-6, V_RAW[trd].mean(0), V_RAW[trd].std(0) + 1e-6)
        F[alld] = (F_RAW[alld] - st[0]) / st[1]; V[alld] = (V_RAW[alld] - st[2]) / st[3]
        if d == 0:
            ch = st
    return F, V, ch
class Branch(nn.Module):
    def __init__(self, d_in, d=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(0.5))
    def forward(self, x):
        return Fn.normalize(self.net(x), dim=1)
class FOP(nn.Module):
    def __init__(self, n_class, d_face, d_voice, d=128):
        super().__init__()
        self.face, self.voice = Branch(d_face, d), Branch(d_voice, d)
        self.att = nn.Sequential(nn.Linear(2 * d, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Linear(d, d))
        self.cls = nn.Linear(d, n_class)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.constant_(m.bias, 0.01)
    def embed(self, xf, xv):
        f, v = self.face(xf), self.voice(xv)
        return torch.tanh(f), torch.tanh(v), f, v
    def forward(self, xf, xv):
        ft, vt, f, v = self.embed(xf, xv)
        a = torch.sigmoid(self.att(torch.cat([f, v], 1)))
        fused = ft * a + (1 - a) * vt
        return fused, self.cls(fused)
def opl_loss(feat, lab):
    feat = Fn.normalize(feat, dim=1); dot = feat @ feat.t()
    same = (lab[:, None] == lab[None, :]).float()
    pos = same - torch.eye(len(lab), device=feat.device); neg = 1 - same
    return (1 - (pos * dot).sum() / (pos.sum() + 1e-6)) + 0.7 * (neg * dot.abs()).sum() / (neg.sum() + 1e-6)
def eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr)); return (fpr[i] + fnr[i]) / 2 * 100
@torch.no_grad()
def same_score(model, xf, xv):
    model.eval(); mf, sf_, mv, sv = model.norm
    ft, vt, _, _ = model.embed((xf - mf) / sf_, (xv - mv) / sv)
    return -(ft - vt).norm(dim=1).cpu().numpy()
def make_pairs(idx, n=2000, same_gender_neg=False, seed=0):
    rng = np.random.default_rng(seed)
    spk_ = np.unique(y[idx]); rows_ = {s: idx[y[idx] == s] for s in spk_}
    V, F, L = [], [], []
    for _ in range(n):
        s = rng.choice(spk_); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[s])); L.append(1)
        s = rng.choice(spk_)
        cand = [t for t in spk_ if t != s and (not same_gender_neg or id_gender[t] == id_gender[s])] or [t for t in spk_ if t != s]
        t = rng.choice(cand); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[t])); L.append(0)
    return np.array(V), np.array(F), np.array(L)
K = 5
rng = np.random.default_rng(42)
folds = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rng.shuffle(members)
    for i, c in enumerate(members):
        folds[i % K].append(c)
FOLD_DATA = []
for k in range(K):
    val_ids = set(folds[k])
    va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_DATA.append((tr_idx, [make_pairs(va_idx, seed=k), make_pairs(va_idx, same_gender_neg=True, seed=100 + k)]))

SEEDS = [1, 2, 3]
P_ID, K_CLIP, ALPHA, LR, MAX_EP, PATIENCE, STEPS = 16, 8, 2.0, 1e-4, 100, 15, 60
def train_variant(tr_idx, ext_rows, in_frac, vsets=None, seed=1, max_epochs=MAX_EP):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    all_tr = np.concatenate([tr_idx, ext_rows])
    F, V, ch = domain_normalize(all_tr)
    XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    cl = np.unique(Y_ALL[all_tr]); rm = {c: i for i, c in enumerate(cl)}
    ytr_np = np.array([rm[c] for c in Y_ALL[all_tr]]); ytr = torch.tensor(ytr_np, device=dev)
    cls_in = np.zeros(len(cl), bool); cls_in[np.unique(ytr_np[D_ALL[all_tr] < 2])] = True
    in_cls, out_cls = np.where(cls_in)[0], np.where(~cls_in)[0]; n_in = int(round(P_ID * in_frac))
    pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(cl))]
    tr = torch.tensor(all_tr, device=dev)
    model = FOP(len(cl), XF.shape[1], XV.shape[1]).to(dev)
    model.norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6)
    mf, sf_, mv, sv = model.norm
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01)
    best, best_ep, best_state, wait = 1e9, max_epochs, None, 0
    for ep in range(1, max_epochs + 1):
        model.train()
        for _ in range(STEPS):
            if len(out_cls) and in_frac < 1:
                chosen = np.r_[rng.choice(in_cls, n_in, replace=False), rng.choice(out_cls, P_ID - n_in, replace=False)]
            else:
                chosen = rng.choice(len(cl), P_ID, replace=False)
            j = torch.tensor(np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen]), device=dev)
            fused, logits = model((XF[tr[j]] - mf) / sf_, (XV[tr[j]] - mv) / sv)
            loss = Fn.cross_entropy(logits, ytr[j]) + ALPHA * opl_loss(fused, ytr[j])
            opt.zero_grad(); loss.backward(); opt.step()
        if vsets is None:
            continue
        score = np.mean([eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_ep, XF, XV, ch

VARIANTS = {'N4x_mavall': ('mav_all', 1.0), 'B1x_mavall_vox_bal75': ('mav_all+vox', 0.75)}
CSV = os.path.join(RESDIR, 'nb14_cv.csv')
done = pd.read_csv(CSV) if os.path.exists(CSV) else pd.DataFrame(columns=['variant', 'seed', 'fold'])
t0 = time.time()
for name, (ext, frac) in VARIANTS.items():
    for seed in SEEDS:
        for k in range(K):
            if ((done.variant == name) & (done.seed == seed) & (done.fold == k)).any():
                continue
            tr_idx, vsets = FOLD_DATA[k]
            model, best_ep, XF, XV, _ = train_variant(tr_idx, ROWS[ext], frac, vsets, seed=seed)
            e_std, e_gen = [eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets]
            row = pd.DataFrame([dict(variant=name, seed=seed, fold=k, best_ep=best_ep, std=e_std, gender=e_gen)])
            row.to_csv(CSV, mode='a', header=not os.path.exists(CSV), index=False)
            done = pd.concat([done, row])
            print('%-22s seed %d fold %d | ep %3d | std %.2f | gender %.2f | %.0f min' % (name, seed, k, best_ep, e_std, e_gen, (time.time() - t0) / 60))
R = pd.read_csv(CSV)
R['overall'] = (R['std'] + R['gender']) / 2
tab = R.groupby(['variant', 'seed'])[['std', 'gender', 'overall']].mean().groupby('variant').mean().round(2)
note('WITH MAV-Celeb v2 + v1 held-out (CV 5-fold x 3 seeds) | references: N4 26.98, B1 25.99')
note(tab.to_string())

In [ ]:
# Cell 6 - final models -> ZIPs + saved dev scores; ensemble ZIP (raw-score average, no normalization - FAQ rule)
LANG_FILE = {'English': 'sub_score_v4_English_heard.txt', 'Bangla': 'sub_score_v4_Bangla_unheard.txt'}
SUBDIR = os.path.join(PROJECT, 'submissions'); os.makedirs(SUBDIR, exist_ok=True)
def write_zip(name, scores):
    OUT = '/content/sub_' + name
    for track in ['no_gender', 'gender']:
        os.makedirs(os.path.join(OUT, track), exist_ok=True)
        for lang_, fname in LANG_FILE.items():
            split = track + '_' + lang_
            pair_ids = [l.split()[0] for l in open(find_one('/' + track + '/', lang_ + '_test.txt')) if l.strip()]
            assert len(pair_ids) == len(scores[split]) and np.isfinite(scores[split]).all()
            with open(os.path.join(OUT, track, fname), 'w') as f:
                for pid, v in zip(pair_ids, -scores[split]):        # distance-like: LOWER = same person
                    f.write('%s %.6f\n' % (pid, v))
    zp = os.path.join(SUBDIR, name + '.zip')
    with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as zf:
        for track in ['no_gender', 'gender']:
            for fname in LANG_FILE.values():
                zf.write(os.path.join(OUT, track, fname), arcname=track + '/' + fname)
    note('Submission ZIP saved: ' + zp[len(BASE):])

DEVS = {}
for name, (ext, frac) in VARIANTS.items():
    ep = int(np.median(R[R.variant == name]['best_ep']))
    finals = []
    for s in SEEDS:
        m, _, _, _, ch = train_variant(np.arange(N_CH), ROWS[ext], frac, None, seed=s, max_epochs=ep)
        finals.append(m)
    sc = {}
    for track in ['no_gender', 'gender']:
        for lang_ in LANG_FILE:
            split = track + '_' + lang_
            fx = ((face_of(FEAT['clip'][split]) - ch[0]) / ch[1]).astype(np.float32)          # challenge TRAIN statistics
            vx = ((voice_of({n: FEAT[n][split] for n in VOICE}) - ch[2]) / ch[3]).astype(np.float32)
            xf = torch.tensor(fx, device=dev); xv = torch.tensor(vx, device=dev)
            sc[split] = np.mean([same_score(m, xf, xv) for m in finals], axis=0)
    DEVS[name] = sc
    np.savez(os.path.join(RESDIR, 'devscores_14_%s.npz' % name), **sc)
    write_zip('14_' + name, sc)
ENS = {s: (DEVS['N4x_mavall'][s] + DEVS['B1x_mavall_vox_bal75'][s]) / 2 for s in DEVS['N4x_mavall']}
write_zip('14_ENS_N4x+B1x', ENS)
print('=' * 70)
print('SUMMARY  (copy everything between the lines and paste to Claude)')
print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)